# 3c · The ARIA chat app + structured output

**Core · Live-code · ~25 min**

So far each call to the model has been a one-off — it forgets everything immediately. Real
assistants hold a **conversation**. In this notebook you'll build a chat that remembers, and
then learn a technique that quietly powers the entire second half of the workshop:
**structured output**.

Solution: [`solution/03c_chat_app.ipynb`](solution/03c_chat_app.ipynb).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import chat

ARIA_SYSTEM = (
    "You are ARIA, the calm, concise assistant for the Orbital Mars colony. "
    "Never invent data; ask for confirmation before any physical action."
)

## Step 1 — How memory actually works in a chat

Here's a surprise for many people: **the model itself has no memory.** Each API call is
independent. The "memory" you see in chat apps is an illusion we create by **resending the
entire conversation every turn.**

So our approach is:
1. Keep a growing list called `history` (starting with the system message).
2. Each turn, append the user's message, send the *whole list*, get a reply.
3. Append that reply too, so it's included next time.

Because the previous turns are in the list, the model can "remember" what was said. Complete
the `say()` function below to implement those steps.

In [ ]:
def new_conversation():
    # Start every conversation with ARIA's system instructions.
    return [{"role": "system", "content": ARIA_SYSTEM}]

def say(history, user_text):
    # TODO: implement one conversational turn.
    # 1) add the user's message:      history.append({"role": "user", "content": user_text})
    # 2) get ARIA's reply:            reply = chat(messages=history)
    # 3) add the reply to history:    history.append({"role": "assistant", "content": reply})
    # 4) return both:                 return history, reply
    ...

history = new_conversation()
history, reply = say(history, "CO2 in Module B just hit 1400 ppm.")
print("ARIA:", reply)

# The real test of memory: refer back to something without repeating it.
history, reply = say(history, "What was the module I just mentioned?")
print("ARIA:", reply)

## Step 2 — Structured output: from prose to data

Free-flowing text is great for humans but awkward for **code**. If we want a program (or an
agent) to *act* on ARIA's assessment, we need it in a predictable, machine-readable shape —
typically **JSON**.

We can just *ask* the model to reply with JSON only, and describe the exact keys we want. Here
we ask for three fields:
- `severity` — one of `nominal`, `amber`, `red`
- `action` — a short recommended action
- `rationale` — a one-line reason

We set `temperature=0` so the format is stable. Models sometimes wrap JSON in code fences
(```), so we gently strip those before parsing with `json.loads` (which turns a JSON string
into a Python dictionary).

In [ ]:
ASSESS_SYSTEM = (
    "You are ARIA. Assess the reading and reply with ONLY a JSON object, no prose, "
    'with keys: severity ("nominal"|"amber"|"red"), action (short string), rationale (short string).'
)

def assess(reading_text):
    # TODO:
    # 1) get the model's JSON reply:  raw = chat(reading_text, system=ASSESS_SYSTEM, temperature=0)
    # 2) tidy stray formatting:       raw = raw.strip().strip("`")
    #                                 if raw.startswith("json"): raw = raw[4:].strip()
    # 3) parse into a Python dict:    return json.loads(raw)
    ...

print(assess("O2 is 18.7% and CO2 is 1600 ppm in Module B"))

## Why this matters (a lot)

That `{severity, action, rationale}` dictionary is something **code can branch on**: `if
result["severity"] == "red": raise_alarm()`. In **Module 5**, this same idea — asking the model
for a structured, actionable response — becomes **tool calling**, where the model's structured
output tells our program which function to run. You just built the bridge to agents. 🎉

### 🌱 Stretch (optional)
- Wrap `say()` in a `while True:` loop that reads `input()` for a real terminal chat.
- Make `assess()` robust: if `json.loads` fails, ask the model once more, more firmly, to
  return only valid JSON.

## ✅ Recap

You built a conversation that remembers (by resending history) and made ARIA emit **structured
JSON** a program can act on. Memory + structure are exactly what turn a chatbot into an
*assistant that can do things* — which is where we're headed.

# 3c · The ARIA chat app + structured output

**Core · Live-code · ~25 min**

Build a multi-turn chat that remembers context, then make ARIA return a structured JSON
assessment we can act on programmatically.

> Solution: [`solution/03c_chat_app.ipynb`](solution/03c_chat_app.ipynb).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import chat

ARIA_SYSTEM = (
    "You are ARIA, the calm, concise assistant for the Orbital Mars colony. "
    "Never invent data; ask for confirmation before any physical action."
)

## A conversation that remembers

Multi-turn chat = keep a growing list of messages and send the whole list each turn.

In [ ]:
def new_conversation():
    return [{"role": "system", "content": ARIA_SYSTEM}]

def say(history, user_text):
    # TODO: append the user message, get ARIA's reply with chat(messages=history),
    #       append the assistant reply, and return (history, reply).
    ...

history = new_conversation()
history, reply = say(history, "CO2 in Module B just hit 1400 ppm.")
print("ARIA:", reply)
history, reply = say(history, "What was the module I just mentioned?")  # tests memory
print("ARIA:", reply)

## Structured output → the bridge to tools

Free text is nice for humans; **JSON** is what code (and agents) can act on. Ask ARIA to
assess a reading and return strict JSON.

In [ ]:
ASSESS_SYSTEM = (
    "You are ARIA. Assess the reading and reply with ONLY a JSON object, no prose, "
    'with keys: severity ("nominal"|"amber"|"red"), action (short string), rationale (short string).'
)

def assess(reading_text):
    # TODO: call chat(reading_text, system=ASSESS_SYSTEM, temperature=0)
    #       then json.loads the reply and return the dict.
    #       (Tip: models sometimes wrap JSON in ```; strip backticks if needed.)
    ...

print(assess("O2 is 18.7% and CO2 is 1600 ppm in Module B"))

That `{severity, action, rationale}` shape is exactly what we'll turn into **tool calls**
in Module 5. 🎉

### 🌱 Stretch
- Wrap `say()` in a `while True:` input loop for a real CLI chat.
- Make `assess()` robust: retry once if the JSON doesn't parse.